# F5-TTS fine-tune for the DJ voice bank

Fine-tunes F5-TTS on ~66 min of target-voice training data prepared
by `tools/prepare_training_data.py` and saves the resulting checkpoint
to Google Drive so you can download it locally and render the 150 DJ
lines on your own machine.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** (free tier).
2. Upload `data/training.tar.gz` (the file produced by
   `prepare_training_data.py`) to your Google Drive at
   `MyDrive/dj-voice/training.tar.gz`.
3. Run cells in order. The full run takes ~4–6 hours; keep the tab
   open. If you get disconnected, the checkpoint cell at the end
   will save progress to Drive periodically — restart and resume.

Total free T4 session limit is ~12 hours, so we have headroom.

## 1. Verify GPU

If this prints `Tesla T4` (or A100/V100) you're good. If it says
`device='cpu'` or fails, change the runtime type via the Runtime
menu and re-run.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Install F5-TTS

Clones the official repo and installs in editable mode so the
training scripts under `src/f5_tts/train/` are importable.

In [ ]:
!apt-get -qq install ffmpeg
!git clone --depth 1 https://github.com/SWivid/F5-TTS.git /content/F5-TTS
%cd /content/F5-TTS
!pip install -q -e .
!pip install -q accelerate

## 3. Mount Drive and unpack training data

Expects `MyDrive/dj-voice/training.tar.gz`. The unpacked layout is:

```
/content/data/training/
  metadata.csv
  wavs/0001.wav … 0862.wav
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
from pathlib import Path

src = Path('/content/drive/MyDrive/dj-voice/training.tar.gz')
assert src.exists(), f'Upload training.tar.gz to {src} first.'

out = Path('/content/data')
out.mkdir(exist_ok=True)
subprocess.run(['tar', '-xzf', str(src), '-C', str(out)], check=True)

n_wavs = len(list((out / 'training' / 'wavs').glob('*.wav')))
print(f'Extracted {n_wavs} wavs')
assert n_wavs > 100, 'Expected hundreds of wavs — check the archive.'

## 4. Resample to 24 kHz + write F5-TTS metadata.csv

F5-TTS trains at 24 kHz; our prep script wrote 16 kHz to keep
Whisper happy. We resample once here and write the metadata in the
format `prepare_csv_wavs.py` expects: `audio_path|text` per line,
no header. Resampling 862 clips on the T4's CPU takes ~2 min.

In [ ]:
import csv
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

src_dir = Path('/content/data/training')
ds_dir = Path('/content/F5-TTS/data/dj_voice')
wav_out = ds_dir / 'wavs'
wav_out.mkdir(parents=True, exist_ok=True)

def resample(wav):
    out_wav = wav_out / wav.name
    if out_wav.exists() and out_wav.stat().st_size > 0:
        return
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', str(wav), '-ar', '24000', '-ac', '1',
        '-c:a', 'pcm_s16le', str(out_wav),
    ], check=True)

wavs = sorted((src_dir / 'wavs').glob('*.wav'))
with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(resample, wavs))
print(f'Resampled {len(wavs)} clips to 24 kHz at {wav_out}')

# F5-TTS' prepare_csv_wavs.py wants:
#   - the literal header `audio_file|text`
#   - absolute paths in the audio_file column
meta_in = src_dir / 'metadata.csv'
meta_out = ds_dir / 'metadata.csv'
with meta_in.open(encoding='utf-8') as fin, meta_out.open('w', encoding='utf-8') as fout:
    reader = csv.reader(fin, delimiter='|')
    next(reader)  # skip the source header row
    fout.write('audio_file|text\n')
    n = 0
    for clip_id, text, _dur in reader:
        abs_path = wav_out / f'{clip_id}.wav'
        fout.write(f'{abs_path}|{text}\n')
        n += 1
print(f'Wrote {n} rows to {meta_out}')

## 5. Build F5-TTS dataset arrow files + fetch the vocab

`prepare_csv_wavs.py` writes `data/dj_voice_pinyin/raw.arrow` and
`duration.json` from your csv. Then we download the matching
`vocab.txt` from HuggingFace — F5-TTS' fine-tune wants the same
character/pinyin vocabulary the base model was trained with.

In [ ]:
%cd /content/F5-TTS
!python src/f5_tts/train/datasets/prepare_csv_wavs.py data/dj_voice/metadata.csv data/dj_voice_pinyin

import os, urllib.request
out_dir = '/content/F5-TTS/data/dj_voice_pinyin'
os.makedirs(out_dir, exist_ok=True)
vocab = f'{out_dir}/vocab.txt'
url = 'https://huggingface.co/SWivid/F5-TTS/resolve/main/F5TTS_v1_Base/vocab.txt'
urllib.request.urlretrieve(url, vocab)
print('vocab size:', os.path.getsize(vocab), 'bytes')

## 6. Fine-tune

Loads the official `F5TTS_v1_Base` checkpoint (downloaded
automatically on first run) and fine-tunes on the dj_voice
dataset. Hyperparameters tuned for ~66 min of training audio:

- `learning_rate 1e-5` — small LR is critical for fine-tune; the
  base model is already good and we only nudge it.
- `batch_size_per_gpu 3200` (frame batches) — fits in 15 GB T4
  VRAM with headroom.
- `epochs 30` — ~3 hours on T4.
- `save_per_updates 1000` — checkpoint every ~1000 steps so a
  Colab disconnect doesn't lose everything.

If the loss stops decreasing well before epoch 30, you can stop
early and use the latest checkpoint.

In [ ]:
%cd /content/F5-TTS
!accelerate launch src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_v1_Base \
    --dataset_name dj_voice \
    --finetune \
    --learning_rate 1e-5 \
    --batch_size_per_gpu 3200 \
    --batch_size_type frame \
    --max_samples 64 \
    --grad_accumulation_steps 1 \
    --max_grad_norm 1.0 \
    --epochs 30 \
    --num_warmup_updates 200 \
    --save_per_updates 1000 \
    --last_per_updates 500

## 7. Sanity-check the fine-tune

Render a single short line from a known DJ-voice prompt to confirm
the checkpoint sounds like the target voice. The generated wav
writes to `/content/sanity.wav`.

In [ ]:
import glob
ckpts = sorted(glob.glob('/content/F5-TTS/ckpts/dj_voice/model_*.pt'))
assert ckpts, 'No checkpoint found in ckpts/dj_voice/.'
ckpt = ckpts[-1]
print('Using checkpoint:', ckpt)

ref_wav = '/content/F5-TTS/data/dj_voice/wavs/0001.wav'
with open('/content/F5-TTS/data/dj_voice/metadata.csv', encoding='utf-8') as f:
    ref_text = f.readline().split('|', 1)[1].strip()

!f5-tts_infer-cli \
    --model F5TTS_v1_Base \
    --ckpt_file "{ckpt}" \
    --ref_audio "{ref_wav}" \
    --ref_text "{ref_text}" \
    --gen_text "Take a breath. The set's about to begin." \
    --output_dir /content/sanity_out

from IPython.display import Audio, display
import glob as _g
out_wavs = _g.glob('/content/sanity_out/*.wav')
print('Output:', out_wavs)
if out_wavs:
    display(Audio(out_wavs[0]))

## 8. Save checkpoint + reference clip to Drive

Copies the final checkpoint and a representative reference WAV
(needed at inference time to anchor the voice) to your Drive so
you can download them and render the 150 DJ lines locally.

In [ ]:
import shutil
out_dir = Path('/content/drive/MyDrive/dj-voice')
out_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(ckpt, out_dir / 'dj_voice_finetuned.pt')
shutil.copy(ref_wav, out_dir / 'reference.wav')

with open('/content/F5-TTS/data/dj_voice/metadata.csv', encoding='utf-8') as f:
    first_text = f.readline().split('|', 1)[1].strip()
(out_dir / 'reference.txt').write_text(first_text, encoding='utf-8')

print('Saved to Google Drive → MyDrive/dj-voice/:')
for p in out_dir.iterdir():
    print(' ', p.name, '-', round(p.stat().st_size / 1e6, 1), 'MB')

## Done

Next steps on your local machine:

1. Download from Drive → `MyDrive/dj-voice/`:
   - `dj_voice_finetuned.pt`
   - `reference.wav`
   - `reference.txt`
2. Place them at `<repo>/data/finetuned/`.
3. Run `tools/render_dj_voice_bank.py` — produces the 150 `.opus`
   files in the bank folder layout matching `manifest.seed.json`.